# E1 fresh JSON receiving of public reconstruction and rescoring

Owner: uibcdf/dockingmt#78. Read the completed native producer in a fresh process.
This notebook performs record admission, public JSON roundtrips and saved-result
audits only. It performs no molecular, mechanics, preparation, scoring, search,
ordering or visualization operation. MolSysMT may load during DockingMT bootstrap.

Set `DOCKINGMT_E1_COMPOSITION_OUTPUT` to the native output directory,
`DOCKINGMT_E1_COMPOSITION_MANIFEST_SHA256` to its independently registered raw
manifest digest, and `DOCKINGMT_E1_COMPOSITION_READER_OUTPUT` to a new, separate
reader directory. The manifest must bind the source registration through
`registration_sha256`; its `registration.json` is either a listed input or in
the enclosing attempt directory. Stored and compressed/decoded identities are
checked before reading. No file is written into the native population.

Normal and pm/fs reader policies receive the same two-pose population; these
are two reader repetitions, not independent molecular cases. Original docking
scores/ranks/backend history remain distinct from the two fixed-score histories.
The reader does not admit a new comparison policy or complete E1 acceptance.

In [ ]:
import gzip
import hashlib
import importlib.metadata as metadata
import json
import math
import os
import platform
import re
import sys
from copy import deepcopy
from pathlib import Path, PurePosixPath

native_root = Path(os.environ['DOCKINGMT_E1_COMPOSITION_OUTPUT']).resolve()
expected_manifest_sha256 = os.environ['DOCKINGMT_E1_COMPOSITION_MANIFEST_SHA256']
output = Path(os.environ['DOCKINGMT_E1_COMPOSITION_READER_OUTPUT']).resolve()
assert re.fullmatch(r'[0-9a-f]{64}', expected_manifest_sha256)
assert not output.is_relative_to(native_root)
assert not native_root.is_relative_to(output)
assert not output.exists()


def digest(data):
    return hashlib.sha256(data).hexdigest()


def reject_constant(value):
    raise ValueError('Nonfinite JSON constant: ' + value)


def decode_json(data):
    return json.loads(data, parse_constant=reject_constant)


def relative_name(value):
    assert isinstance(value, str) and value
    path = PurePosixPath(value)
    assert not path.is_absolute() and '..' not in path.parts
    assert str(path) == value
    return value


manifest_path = native_root / 'manifest.json'
manifest_bytes = manifest_path.read_bytes()
assert digest(manifest_bytes) == expected_manifest_sha256
manifest = decode_json(manifest_bytes)
assert manifest['owner'] == 'uibcdf/dockingmt#78'
assert isinstance(manifest['files'], list) and manifest['files']
stored_inputs = {}
documents = {}
for entry in manifest['files']:
    name = relative_name(entry['path'])
    logical_name = relative_name(entry.get('original_path', name))
    assert name not in stored_inputs and logical_name not in documents
    path = (native_root / name).resolve()
    assert path.is_relative_to(native_root) and path != manifest_path
    data = path.read_bytes()
    assert type(entry['bytes']) is int and entry['bytes'] == len(data)
    assert digest(data) == entry['sha256']
    stored_inputs[name] = dict(path=path, bytes=len(data), sha256=digest(data))
    decoded = data
    if name.endswith('.gz'):
        decoded = gzip.decompress(data)
        assert len(decoded) == entry['decoded_bytes']
        assert digest(decoded) == entry['decoded_sha256']
    documents[logical_name] = decoded

registration_sha256 = manifest['registration_sha256']
assert re.fullmatch(r'[0-9a-f]{64}', registration_sha256)
if 'registration.json' in documents:
    registration_bytes = documents['registration.json']
    registration_path = None
else:
    registration_path = native_root.parent / 'registration.json'
    registration_bytes = registration_path.read_bytes()
assert digest(registration_bytes) == registration_sha256
source_registration = decode_json(registration_bytes)
assert source_registration['owner'] == 'uibcdf/dockingmt#78'

required_documents = (
    'user/source_result.json',
    'user/updated_result.json',
    'user/updated_audit.json',
    'native_receipt.json',
)
assert all(name in documents for name in required_documents)
source_record = decode_json(documents['user/source_result.json'])
updated_record = decode_json(documents['user/updated_result.json'])
saved_updated_audit = decode_json(documents['user/updated_audit.json'])
native_receipt = decode_json(documents['native_receipt.json'])
assert native_receipt['owner'] == 'uibcdf/dockingmt#78'
record_inputs_before = deepcopy(
    (source_record, updated_record, saved_updated_audit, native_receipt)
)
output.mkdir(parents=True, exist_ok=False)


def save(path, value):
    with path.open('x') as stream:
        json.dump(value, stream, sort_keys=True, indent=2, allow_nan=False)
        stream.write('\n')

In [ ]:
import pyunitwizard as puw

import dockingmt as dmt

installed_check = globals().get('registered_installed_check')
if installed_check is not None:
    assert callable(installed_check)
    installed_check('before-composition-json-receiving')

budgets = dict(
    reader_policies=2,
    from_dict=8,
    audit_result=4,
    result_roundtrips=4,
    detachment_controls=4,
)
counts = dict.fromkeys(budgets, 0)


def consume(name):
    assert counts[name] < budgets[name], name
    counts[name] += 1


original_score_names = {'vina', 'inter', 'intra', 'torsion'}
evaluation_methods = [('vina_fixed', 'vina'), ('vinardo_fixed', 'vinardo')]
component_suffixes = (
    '',
    '.lig_inter',
    '.flex_inter',
    '.other_inter',
    '.flex_intra',
    '.lig_intra',
    '.torsions',
    '.lig_intra_best_pose',
)


def finite_scores(scores):
    assert isinstance(scores, dict)
    assert all(
        type(value) in (int, float) and math.isfinite(value)
        for value in scores.values()
    )


def check_preservation(source, updated):
    assert set(source) == set(updated)
    assert len(source['poses']) == len(updated['poses']) == 2
    for field in source:
        if field != 'poses':
            assert source[field] == updated[field], field
    for field in ('problem_info', 'protocol_info', 'provenance'):
        assert field in source and source[field] == updated[field], field
    original_ranking = source['provenance']['ranking_history']
    assert isinstance(original_ranking, list) and original_ranking
    assert any(entry.get('origin') == 'backend' for entry in original_ranking)
    assert updated['provenance']['ranking_history'] == original_ranking
    assert [pose['rank'] for pose in source['poses']] == [1, 2]
    assert len({pose['pose_id'] for pose in source['poses']}) == 2
    rows = []
    for position, (prior, current) in enumerate(
        zip(source['poses'], updated['poses']), 1
    ):
        assert set(prior) == set(current)
        for field in prior:
            if field not in ('scores', 'metadata'):
                assert prior[field] == current[field], (position, field)
        assert set(prior['scores']) == original_score_names
        finite_scores(prior['scores'])
        finite_scores(current['scores'])
        assert len(current['scores']) == 20
        assert all(
            current['scores'][name] == value for name, value in prior['scores'].items()
        )
        previous_metadata = prior['metadata']
        current_metadata = current['metadata']
        allowed_changes = {'score_definitions', 'scoring_history'}
        assert (
            set(previous_metadata) - allowed_changes
            == set(current_metadata) - allowed_changes
        )
        for field in set(previous_metadata) - allowed_changes:
            assert previous_metadata[field] == current_metadata[field], (
                position,
                field,
            )
        assert not previous_metadata.get('scoring_history', [])
        previous_definitions = previous_metadata['score_definitions']
        current_definitions = current_metadata['score_definitions']
        assert set(previous_definitions) == original_score_names
        assert len(current_definitions) == 20
        assert all(
            current_definitions[name] == value
            for name, value in previous_definitions.items()
        )
        histories = current_metadata['scoring_history']
        assert len(histories) == 2
        assert [entry['score_name'] for entry in histories] == [
            name for name, _ in evaluation_methods
        ]
        added_names = set()
        for evaluation, (name, method) in zip(histories, evaluation_methods):
            names = {name + suffix for suffix in component_suffixes}
            added_names.update(names)
            assert (
                evaluation['operation'] == 'score' and evaluation['backend'] == 'vina'
            )
            assert evaluation['applied_parameters']['scoring'] == method
            assert set(evaluation['scores']) == names
            assert set(evaluation['score_definitions']) == names
            assert evaluation['scores'] == {
                key: current['scores'][key] for key in names
            }
            assert evaluation['score_definitions'] == {
                key: current_definitions[key] for key in names
            }
            assert evaluation['reference_energy'] == dict(
                schema_version='1.0',
                policy='native_current_conformation',
                score_name=name + '.lig_intra_best_pose',
                value=current['scores'][name + '.lig_intra_best_pose'],
                unit='kcal/mol',
            )
        assert set(current['scores']) == original_score_names | added_names
        assert set(current_definitions) == original_score_names | added_names
        rows.append(
            dict(
                source_position=position,
                pose_id=prior['pose_id'],
                rank=prior['rank'],
                original_scores=4,
                retained_new_scores=16,
                fixed_score_histories=2,
                original_fields_preserved=True,
            )
        )
    return rows


wire_preservation = check_preservation(source_record, updated_record)
assert saved_updated_audit['status'] == 'consistent'
assert not any(
    check['status'] == 'inconsistent' for check in saved_updated_audit['checks']
)

In [ ]:
caller_policy = puw.configure.report()
profiles = [
    (
        'normal',
        [
            'nm',
            'ps',
            'kelvin',
            'mole',
            'dalton',
            'e',
            'kJ/mol',
            'kJ/(mol*nm)',
            'kJ/(mol*nm**2)',
            'radians',
        ],
    ),
    (
        'pm_fs',
        [
            'pm',
            'fs',
            'kelvin',
            'mole',
            'dalton',
            'coulomb',
            'kJ/mol',
            'kJ/(mol*pm)',
            'kJ/(mol*pm**2)',
            'radians',
        ],
    ),
]
observations = []
for profile, standard_units in profiles:
    consume('reader_policies')
    directory = output / profile
    directory.mkdir()
    with puw.context(standard_units=standard_units):
        active_policy = puw.configure.report()
        canonical_records = {}
        audit_summaries = {}
        for label, original in [('source', source_record), ('updated', updated_record)]:
            consume('from_dict')
            model = dmt.DockingResult.from_dict(deepcopy(original))
            canonical_before = model.to_dict()
            consume('audit_result')
            audit = dmt.audit_result(deepcopy(original))
            save(directory / (label + '_audit.json'), audit)
            assert audit['status'] == 'consistent', (profile, label, audit)
            assert not any(
                check['status'] == 'inconsistent' for check in audit['checks']
            )
            if label == 'updated':
                assert audit == saved_updated_audit, profile
            canonical_wire = json.loads(json.dumps(canonical_before, allow_nan=False))
            consume('from_dict')
            restored = dmt.DockingResult.from_dict(canonical_wire)
            consume('result_roundtrips')
            assert restored.to_dict() == canonical_before
            assert model.to_dict() == canonical_before
            detached = model.to_dict()
            detached['provenance']['reader_detachment_control'] = True
            assert model.to_dict() == canonical_before
            consume('detachment_controls')
            canonical_records[label] = canonical_before
            audit_summaries[label] = dict(
                status=audit['status'], checks=len(audit['checks'])
            )
            assert puw.configure.report() == active_policy
        canonical_preservation = check_preservation(
            canonical_records['source'], canonical_records['updated']
        )
        assert canonical_preservation == wire_preservation
        assert (
            source_record,
            updated_record,
            saved_updated_audit,
            native_receipt,
        ) == record_inputs_before
        observations.append(
            dict(
                profile=profile,
                poses=2,
                original_scores_per_pose=4,
                updated_scores_per_pose=20,
                fixed_histories_per_pose=2,
                preserved_backend_ranking_history=True,
                audits=audit_summaries,
                canonical_roundtrips=True,
                detached_snapshots=True,
                input_records_unchanged=True,
            )
        )
    assert puw.configure.report() == caller_policy

In [ ]:
assert counts == budgets
assert len(observations) == 2
assert (
    source_record,
    updated_record,
    saved_updated_audit,
    native_receipt,
) == record_inputs_before
assert manifest_path.read_bytes() == manifest_bytes
for entry in stored_inputs.values():
    data = entry['path'].read_bytes()
    assert len(data) == entry['bytes'] and digest(data) == entry['sha256']
if registration_path is not None:
    assert registration_path.read_bytes() == registration_bytes
assert not any(
    name == forbidden or name.startswith(forbidden + '.')
    for forbidden in ('vina', 'rdkit', 'molsysviewer')
    for name in sys.modules
)
assert puw.configure.report() == caller_policy
if installed_check is not None:
    installed_check('after-composition-json-receiving')
receipt = dict(
    schema='dockingmt.e1_public_composition_reader@1',
    owner='uibcdf/dockingmt#78',
    python=platform.python_version(),
    versions={
        name: metadata.version(name)
        for name in ('dockingmt', 'molsysmt', 'pyunitwizard')
    },
    source_manifest_sha256=expected_manifest_sha256,
    source_registration_sha256=registration_sha256,
    native_registration_owner=source_registration['owner'],
    verified_stored_files=len(stored_inputs),
    counts=counts,
    reader_policy_repetitions=2,
    distinct_native_populations=1,
    profiles=observations,
    preserved_poses=wire_preservation,
    source_files_unchanged=True,
    original_docking_ranking_preserved=True,
    new_empirical_values=0,
    new_molecular_operations=0,
    new_vina_calls=0,
    excluded_operation_basis='Inspected record-only public result admission, audits and quantity serialization; no molecular/native counter hooks.',
    optional_engine_or_viewer_modules_present=False,
    installed_origin_hook_used=installed_check is not None,
    positive_comparison_policy_received=False,
    full_e1_accepted=False,
    scope='Fresh JSON receiving of preserved docking observations and additive fixed-score histories; no molecular restoration, new computation, ordering or full E1 acceptance.',
)
save(output / 'receipt.json', receipt)
print(
    json.dumps(
        dict(
            reader_policies=2,
            poses=2,
            audit_result=counts['audit_result'],
            preserved_original_scores=8,
            retained_fixed_components=32,
            new_empirical_values=0,
        )
    )
)